# 02 — Clean -> Fingerprint -> Features -> Proxy drought label
Output: `clean_daily.csv`, `fingerprint.csv`, `features_daily.csv` (the model table).

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd, numpy as np
from config import RAW_CSV_GLOB, OUTPUT_DIR, DROP_COLS, CLIP_RANGES, DOY_WINDOW, STRESS_LABEL_QUANTILE
from loader import load_all_data

df = load_all_data(RAW_CSV_GLOB)
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
for c,(lo,hi) in CLIP_RANGES.items():          # physically-impossible values -> NaN
    if c in df.columns: df.loc[~df[c].between(lo,hi), c] = np.nan
df = df.interpolate(method='time', limit=8)     # fill short gaps only (8 x 15min = 2h)

hourly = df.resample('1h').mean(numeric_only=True)
rain_h = df[['rain1','rain2']].clip(lower=0).resample('1h').sum()
hourly['rain_mm'] = rain_h.mean(axis=1)

#Daily aggregates: the modelling unit
d = pd.DataFrame({
    'temp_max': hourly['temp_sht'].resample('1D').max(),
    'temp_mean': hourly['temp_sht'].resample('1D').mean(),
    'humidity_mean': hourly['humidity_sht'].resample('1D').mean(),
    'humidity_min': hourly['humidity_sht'].resample('1D').min(),
    'pressure_mean': hourly['press_bmx'].resample('1D').mean(),
    'light_mean': hourly['light_vis'].resample('1D').mean(),
    'wind_mean': hourly['wind_spd'].resample('1D').mean(),
    'rain_mm': hourly['rain_mm'].resample('1D').sum(),
})
d.index.name = 'date'
d.to_csv(f'{OUTPUT_DIR}/clean_daily.csv')
print(f'Daily rows: {len(d)}  |  rain days (>=1mm): {int((d.rain_mm>=1).sum())}')

  loaded 3DFEWSNET_SiteJKUAT_KenyaKiambuJKUATIOTAWS-Conduti@Empathy1.csv: 7,060 rows  2026-08-28 00:00:25+00:00 -> 2026-09-01 23:58:31+00:00
  loaded weatherdata (1).csv: 3,040 rows  2026-07-14 00:00:02+00:00 -> 2026-08-14 23:53:20+00:00
  loaded weatherdata (2).csv: 2,945 rows  2026-06-14 00:01:00+00:00 -> 2026-07-14 23:49:55+00:00
  loaded weatherdata (3).csv: 3,040 rows  2026-05-14 00:00:18+00:00 -> 2026-06-14 23:49:37+00:00
  loaded weatherdata (4).csv: 2,943 rows  2026-04-14 00:00:00+00:00 -> 2026-05-14 23:47:10+00:00
  loaded weatherdata (5).csv: 3,040 rows  2026-03-14 00:00:52+00:00 -> 2026-04-14 23:49:10+00:00
  loaded weatherdata (6).csv: 2,751 rows  2026-02-14 00:00:11+00:00 -> 2026-03-14 23:51:30+00:00
  loaded weatherdata (7).csv: 3,036 rows  2026-01-14 00:01:00+00:00 -> 2026-02-14 23:48:07+00:00
  loaded weatherdata (8).csv: 3,032 rows  2025-12-14 00:00:31+00:00 -> 2026-01-14 23:54:03+00:00
TOTAL: 30,223 rows | 2025-12-14 00:00:31+00:00 -> 2026-09-01 23:58:31+00:00
Daily r

In [ ]:
# FINGERPRINT: seasonal baseline

# Use a +/- DOY_WINDOW-day *window* around each day-of-year (circular, so Dec/Jan wrap
# correctly) instead of an exact match. A window of neighbouring days gives each period
# plenty of samples even from one season, and will naturally start blending in real
# multi-year matches once you have 12+ months of data.
span_days = (d.index.max() - d.index.min()).days
use_doy = span_days >= 180
vars_ = ['temp_max', 'temp_mean', 'humidity_mean', 'humidity_min', 'rain_mm', 'light_mean']

if use_doy:
    doy = d.index.dayofyear.values
    rows = []
    for p in np.unique(doy):
        dist = np.abs(doy - p)
        dist = np.minimum(dist, 366 - dist)          # circular distance: Jan 2 vs Dec 30 is close, not far
        mask = dist <= DOY_WINDOW
        row = {'period': p}
        for v in vars_:
            row[f'{v}_mean'] = d.loc[mask, v].mean()
            row[f'{v}_std'] = d.loc[mask, v].std()
        rows.append(row)
    fp = pd.DataFrame(rows)
else:
    grp = d.groupby(d.index.month)
    fp = grp[vars_].agg(['mean', 'std']).reset_index()
    fp.columns = [f'{v}_{s}' for v, s in fp.columns]
    fp = fp.rename(columns={fp.columns[0]: 'period'})

# Safety net: if any period still has a NaN/zero std (window too small, sparse data, etc.),
# backfill with the dataset-wide mean/std for that variable so a z-score never comes out
# NaN for every row again, even in edge cases we haven't hit yet.
for v in vars_:
    fp[f'{v}_std'] = fp[f'{v}_std'].replace(0, np.nan).fillna(d[v].std())
    fp[f'{v}_mean'] = fp[f'{v}_mean'].fillna(d[v].mean())

fp.to_csv(f'{OUTPUT_DIR}/fingerprint.csv', index=False)
label = f'day-of-year (+/-{DOY_WINDOW}d window)' if use_doy else 'month'
print(f'Span: {span_days} days -> fingerprint by {label}')
print('fingerprint.csv -> outputs/')
fp


Span: 261 days -> fingerprint by day-of-year (+/-15d window)
fingerprint.csv -> outputs/


,period,temp_max_mean,temp_max_std,temp_mean_mean,temp_mean_std,humidity_mean_mean,humidity_mean_std,humidity_min_mean,humidity_min_std,rain_mm_mean,rain_mm_std,light_mean_mean,light_mean_std
0,1,28.943611,1.629068,21.186169,0.977735,65.458333,8.452312,40.325278,8.816743,0.050000,0.130648,440.224421,33.979088
1,2,28.964444,1.633474,21.177558,0.969120,65.256667,8.472458,40.180278,8.811685,0.050000,0.130648,440.526968,34.024721
2,3,29.023611,1.605829,21.217963,0.993902,64.963368,8.355025,39.986944,8.720622,0.050000,0.130648,441.626620,33.569672
3,4,29.074444,1.607089,21.237384,0.999894,64.712153,8.248485,39.787778,8.705840,0.050000,0.130648,441.603704,33.593182
4,5,29.284722,1.426571,21.340567,0.922517,63.932500,7.513387,38.851667,7.693481,0.040000,0.122051,445.575463,29.743826
...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,361,28.749107,1.583774,21.232453,1.002798,67.874814,7.539992,42.348810,7.475390,0.067857,0.149204,436.609871,33.088188
258,362,28.800000,1.579199,21.256789,0.993411,67.397222,7.838099,41.911782,7.708729,0.065517,0.147057,437.490421,32.836152
259,363,28.739167,1.587103,21.241100,0.979908,67.232824,7.754233,42.015556,7.595950,0.063333,0.144993,436.417824,32.795534
260,364,28.770000,1.583237,21.258368,0.973976,66.791829,7.671786,41.741389,7.580662,0.050000,0.130648,436.362847,32.794829


In [ ]:
# FEATURES: rolling windows, anomalies vs fingerprint, trends, dry spell
# NOTE: min_periods lets a window emit a value before it's fully "full" (e.g. day 15
# of a 30-day rolling window), instead of being NaN until day 30. 
f = d.copy()
f['rain_7d'] = f['rain_mm'].rolling(7, min_periods=4).sum()
f['rain_30d'] = f['rain_mm'].rolling(30, min_periods=15).sum()
f['humidity_7d'] = f['humidity_mean'].rolling(7, min_periods=4).mean()
f['humidity_min_7d'] = f['humidity_min'].rolling(7, min_periods=4).min()
f['tempmax_7d'] = f['temp_max'].rolling(7, min_periods=4).max()
f['light_7d'] = f['light_mean'].rolling(7, min_periods=4).mean()
f['humidity_trend'] = f['humidity_7d'] - f['humidity_mean'].shift(7).rolling(7, min_periods=4).mean()
f['temp_trend'] = f['tempmax_7d'] - f['temp_max'].shift(7).rolling(7, min_periods=4).max()
rain_days = f['rain_mm'] >= 1.0
f['dry_spell'] = rain_days.groupby((~rain_days).cumsum()).cumsum()
# merge fingerprint stats
f['period'] = f.index.dayofyear if use_doy else f.index.month
f = f.merge(fp, on='period', how='left')
for v in ['temp_mean','humidity_mean','rain_mm']:
    m_, s_ = f[f'{v}_mean'], f[f'{v}_std'].replace(0, np.nan)
    f[f'{v}_z'] = (f[v] - m_) / s_
f['rain_deficit_30d'] = (f['rain_mm_mean']*30 - f['rain_30d']).clip(lower=0)

# Diagnostics: see exactly how many rows survive, and why, before it becomes a problem later
_feat_check = ['rain_7d','rain_30d','humidity_7d','humidity_min_7d','tempmax_7d','light_7d',
               'humidity_trend','temp_trend','dry_spell','temp_mean_z','humidity_mean_z',
               'rain_mm_z','rain_deficit_30d']
print(f'Total daily rows: {len(f)}')
print('NaN count per feature column (should be small, not close to total rows):')
print(f[_feat_check].isna().sum())
f.tail(3)


Total daily rows: 262
NaN count per feature column (should be small, not close to total rows):
rain_7d              3
rain_30d            14
humidity_7d         16
humidity_min_7d     16
tempmax_7d          16
light_7d            16
humidity_trend      25
temp_trend          25
dry_spell            0
temp_mean_z         13
humidity_mean_z     13
rain_mm_z            0
rain_deficit_30d    14
dtype: int64


,temp_max,temp_mean,humidity_mean,humidity_min,pressure_mean,light_mean,wind_mean,rain_mm,rain_7d,rain_30d,...,humidity_min_mean,humidity_min_std,rain_mm_mean,rain_mm_std,light_mean_mean,light_mean_std,temp_mean_z,humidity_mean_z,rain_mm_z,rain_deficit_30d
259,25.501667,19.751145,67.454406,40.570000,853.684216,404.516853,0.696075,0.0,0.0,0.0,...,42.03774,10.819684,0.011111,0.047140,394.520209,56.524416,0.316937,0.292188,-0.235702,0.333333
260,21.483051,17.989216,75.643543,59.842373,853.259863,307.650001,0.465092,0.2,0.2,0.2,...,42.03774,10.819684,0.011765,0.048507,394.520209,56.524416,-1.773119,1.431681,3.880570,0.152941
261,25.962069,20.037140,66.408012,42.891525,853.071245,374.842057,0.538747,0.0,0.2,0.2,...,42.03774,10.819684,0.012500,0.050000,394.520209,56.524416,0.656193,0.146586,-0.250000,0.175000


In [ ]:
# PROXY DROUGHT-STRESS LABEL (no ground truth exists -> rule-based index)
# Components: rainfall deficit, low humidity, heat, long dry spell.
# Label = top 20% of stress_score (guarantees class balance even in a fully dry season).
f['comp_rain'] = (f['rain_deficit_30d'] / (f['rain_mm_mean']*30 + 1e-6)).clip(0,1)
f['comp_hum']  = ((-f['humidity_mean_z']) / 3).clip(0,1)
f['comp_heat'] = ((f['temp_mean_z']) / 3).clip(0,1)
f['comp_dry']  = (f['dry_spell'] / 30).clip(0,1)
f['stress_score'] = (0.40*f['comp_rain'] + 0.25*f['comp_hum'] + 0.15*f['comp_heat'] + 0.20*f['comp_dry'])
thr = f['stress_score'].quantile(STRESS_LABEL_QUANTILE)
f['y'] = (f['stress_score'] >= thr).astype(int)
feature_cols = ['rain_7d','rain_30d','humidity_7d','humidity_min_7d','tempmax_7d','light_7d',
                'humidity_trend','temp_trend','dry_spell','temp_mean_z','humidity_mean_z',
                'rain_mm_z','rain_deficit_30d']
model_df = f.dropna(subset=feature_cols + ['y'])
if len(model_df) == 0:
    # Tell you exactly which column(s) are NaN on every row, instead of failing silently
    # three cells later inside sklearn with a cryptic "0 samples" error.
    culprits = f[feature_cols + ['y']].isna().mean().sort_values(ascending=False)
    raise ValueError(
        'model_df is EMPTY after dropna() -- 0 rows will reach notebook 03.\n'
        f'Fraction of rows that are NaN, per column (fix the worst offenders first):\n{culprits}\n'
        'Likely causes: (1) not enough raw days for a 30-day window even with min_periods, '
        '(2) the fingerprint merge in the cell above found no matching period for some/all rows, '
        'or (3) most of the raw data was clipped to NaN by CLIP_RANGES in config.py.'
    )
model_df[feature_cols + ['y','stress_score']].to_csv(f'{OUTPUT_DIR}/features_daily.csv', index_label='date')
print(f'Modelling rows: {len(model_df)} | positives: {model_df.y.mean():.0%} | threshold={thr:.3f}')
print('FEATURE_COLS =', feature_cols)  # copy this list into src/pipeline.py


Modelling rows: 230 | positives: 19% | threshold=0.199
FEATURE_COLS = ['rain_7d', 'rain_30d', 'humidity_7d', 'humidity_min_7d', 'tempmax_7d', 'light_7d', 'humidity_trend', 'temp_trend', 'dry_spell', 'temp_mean_z', 'humidity_mean_z', 'rain_mm_z', 'rain_deficit_30d']
